In [1]:
# parameters for the request
params = dict(
    channel="BHZ",
    starttime="1995-11-14T06:32:55.750000Z",
    endtime="1995-11-14T06:35:55.750000Z",
    # level="channel",
    # includeoverlaps="false",
    nodata=404,
)


In [2]:
# Use ObsPy's RoutingClient to send the request
from obspy.clients.fdsn import RoutingClient

client = RoutingClient(
    "iris-federator",
    debug=False,
    timeout=10,
    # include_providers=["SCEDC"],
    exclude_providers=None,
    credentials=None,
)
# client.get_stations(**params)
st = client.get_waveforms(**params)


/home/seisman/opt/miniconda/lib/python3.10/site-packages/obspy/clients/fdsn/routing/routing_client.py:120: UserWarning: No FDSN services could be discovered at 'http://www.orfeus-eu.org'. This could be due to a temporary service outage or an invalid FDSN service address.It will not be used for routing. Try again later?
  warnings.warn(msg)
/home/seisman/opt/miniconda/lib/python3.10/site-packages/obspy/clients/fdsn/routing/routing_client.py:120: UserWarning: No FDSN services could be discovered at 'https://eida-sc3.infp.ro'. This could be due to a temporary service outage or an invalid FDSN service address.It will not be used for routing. Try again later?
  warnings.warn(msg)


In [6]:
# Define a custom enhanced FederatorRoutingClient
import collections
from obspy.clients.fdsn.client import get_bulk_string
from obspy.clients.fdsn.routing.federator_routing_client import FederatorRoutingClient
from obspy.clients.fdsn.header import FDSNNoDataException


class EnhancedFederatorRoutingClient(FederatorRoutingClient):
    """
    Enhanced FederatorRoutingClient that allows more control over the request.
    """

    def __init__(
        self, debug=False, timeout=120, include_providers=None, exclude_providers=None
    ):
        """
        Initialize the client.

        Parameters
        ----------
        debug : bool, optional
            If True, print out debug information.
        timeout : int, optional
            Timeout in seconds.
        include_providers : list of str, optional
            List of providers to include. If not given, all providers are
            included.
        exclude_providers : list of str, optional
            List of providers to exclude. If not given, no providers are
            excluded.
        """
        super().__init__(
            url="http://service.iris.edu/irisws/fedcatalog/1",
            debug=debug,
            timeout=timeout,
            include_providers=include_providers,
            exclude_providers=exclude_providers,
        )

    def get_available_channels(self, **kwargs):
        """
        Get available channels. Accepts the same parameters as get_stations.
        """
        # convert the six parameters to a bulk list
        bulk = []
        for i in ["network", "station", "location", "channel", "starttime","endtime"]:
            if i not in kwargs or kwargs[i] is None:
                bulk.append("*")
            else:
                bulk.append(kwargs.pop(i)) 
        bulk = [bulk]  # bulk must be a list of lists

        # parameters that should be passed
        params = {k: str(kwargs[k])
                  for k in self.kwargs_of_interest if k in kwargs}
        params["format"] = "request" 

        bulk_str = get_bulk_string(bulk, params)

        # request the available channels
        r = self._download(self._url + "/query", data=bulk_str, content_type="text/plain")
        
        # split the responses for multiple datacenters
        records = self._split_routing_response(
            r.content.decode() if hasattr(r.content, "decode") else r.content,
            service="station")
        # filter requests based on including and excluding providers
        records = self._filter_requests(records)

        if not records:
            raise FDSNNoDataException(
                "Nothing remains to download after the provider "
                "inclusion/exclusion filters have been applied.")

        return records


client = EnhancedFederatorRoutingClient(timeout=10)
# st = client.get_waveforms(**params)
# print(len(st))

records = client.get_available_channels(**params)
from datetime import datetime
print(datetime.now())
inv = client.get_stations(**params, level="station")
print(datetime.now())
inv = client._download_stations(records, level="station")
print(datetime.now())
# st = client._download_waveforms(records)
# print(st)
# client.get_waveforms_bulk(records)


2023-09-11 16:30:16.007307


/home/seisman/opt/miniconda/lib/python3.10/site-packages/obspy/clients/fdsn/routing/routing_client.py:120: UserWarning: No FDSN services could be discovered at 'http://www.orfeus-eu.org'. This could be due to a temporary service outage or an invalid FDSN service address.It will not be used for routing. Try again later?
  warnings.warn(msg)
/home/seisman/opt/miniconda/lib/python3.10/site-packages/obspy/clients/fdsn/routing/routing_client.py:120: UserWarning: No FDSN services could be discovered at 'https://eida-sc3.infp.ro'. This could be due to a temporary service outage or an invalid FDSN service address.It will not be used for routing. Try again later?
  warnings.warn(msg)


2023-09-11 16:30:27.789134


/home/seisman/opt/miniconda/lib/python3.10/site-packages/obspy/clients/fdsn/routing/routing_client.py:120: UserWarning: No FDSN services could be discovered at 'http://www.orfeus-eu.org'. This could be due to a temporary service outage or an invalid FDSN service address.It will not be used for routing. Try again later?
  warnings.warn(msg)
/home/seisman/opt/miniconda/lib/python3.10/site-packages/obspy/clients/fdsn/routing/routing_client.py:120: UserWarning: No FDSN services could be discovered at 'https://eida-sc3.infp.ro'. This could be due to a temporary service outage or an invalid FDSN service address.It will not be used for routing. Try again later?
  warnings.warn(msg)


2023-09-11 16:30:38.679353


In [4]:
for net in inv:
    for sta in net:
        print(sta)

Station BZN (Buzz Northerns Place, Anza, CA, USA)
	Station Code: BZN
	Channel Count: 0/383 (Selected/Total)
	1983-01-20T00:00:00.000000Z - 
	Access: open 
	Latitude: 33.4915, Longitude: -116.6670, Elevation: 1301.0 m
	Available Channels:

Station CRY (Cary Ranch, Anza, CA, USA)
	Station Code: CRY
	Channel Count: 0/437 (Selected/Total)
	1982-10-01T00:00:00.000000Z - 
	Access: open 
	Latitude: 33.5654, Longitude: -116.7373, Elevation: 1128.0 m
	Available Channels:

Station FRD (Ford Ranch, Anza, CA, USA)
	Station Code: FRD
	Channel Count: 0/310 (Selected/Total)
	1982-10-01T00:00:00.000000Z - 
	Access: open 
	Latitude: 33.4947, Longitude: -116.6022, Elevation: 1164.0 m
	Available Channels:

Station KNW (Keenwild Fire Station, Mountain Center, CA, USA)
	Station Code: KNW
	Channel Count: 0/353 (Selected/Total)
	1982-10-01T00:00:00.000000Z - 
	Access: open 
	Latitude: 33.7141, Longitude: -116.7119, Elevation: 1507.0 m
	Available Channels:

Station LVA2 (Lost Valley Scout Camp, CA, USA)
	Stat

In [ ]:
from obspy.clients.fdsn import Client
from obspy.clients.fdsn.header import (
    FDSNNoServiceException,
    FDSNNoDataException,
    FDSNTimeoutException,
)
from pathlib import Path


Path("stations").mkdir(exist_ok=True, parents=True)
Path("mseed").mkdir(exist_ok=True, parents=True)


from multiprocessing.dummy import Pool as ThreadPool


def _try_download_bulk(datacenter, bulk):
    try:
        client = Client(datacenter, timeout=120)
    except FDSNNoServiceException:
        print("No service for", datacenter)
        return None, None
    try:
        inv = client.get_stations_bulk(bulk=records[datacenter], level="response")
        st = client.get_waveforms_bulk(bulk=records[datacenter])
        print(len(st))
        return inv, st
    except (
        FDSNNoDataException,
        FDSNTimeoutException,
        FDSNBadGatewayException,
        ValueError,
    ):
        return None, None


pool = ThreadPool(processes=len(records.keys()))
args = [(datacenter, records[datacenter]) for datacenter in records.keys()]
pool.starmap(_try_download_bulk, args)
pool.close()
